# 03 - Phân cụm EM (Expectation Maximization)
## EM Clustering with Gaussian Mixture Model

Mục tiêu:
- Giảm chiều TF-IDF bằng TruncatedSVD
- Thử nhiều giá trị k, đánh giá BIC, AIC, Silhouette
- Chọn k tối ưu và train GMM
- Hiển thị top keywords và vẽ PCA/t-SNE

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from scipy.sparse import load_npz
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import normalize
from sklearn.manifold import TSNE

os.makedirs('../report/images', exist_ok=True)
plt.style.use('seaborn-v0_8-darkgrid')
print("Ready!")

## 3.1 Load Data

In [ ]:
# Load TF-IDF matrix và data
tfidf_matrix = load_npz('../data/processed/tfidf_matrix.npz')
vectorizer = joblib.load('../models/tfidf_vectorizer.pkl')
df = pd.read_csv('../data/processed/unified_data.csv')

print(f"TF-IDF matrix: {tfidf_matrix.shape}")
print(f"Dataset: {df.shape}")

## 3.2 Giảm chiều bằng TruncatedSVD (LSA)

In [ ]:
# SVD reduction
n_components = 100
print(f"Reducing to {n_components} components...")

svd = TruncatedSVD(n_components=n_components, random_state=42)
X_reduced = svd.fit_transform(tfidf_matrix)
X_reduced = normalize(X_reduced)

explained_var = svd.explained_variance_ratio_.sum()
print(f"Explained variance: {explained_var:.3f} ({explained_var:.1%})")
print(f"Reduced shape: {X_reduced.shape}")

# Plot explained variance
fig, ax = plt.subplots(figsize=(10, 5))
cumsum = np.cumsum(svd.explained_variance_ratio_)
ax.plot(range(1, len(cumsum)+1), cumsum, 'b-', lw=2)
ax.axhline(0.8, color='red', linestyle='--', label='80% threshold')
ax.set_title('Cumulative Explained Variance (SVD)', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Components')
ax.set_ylabel('Cumulative Explained Variance')
ax.legend()
plt.tight_layout()
plt.savefig('../report/images/nb03_svd_variance.png', dpi=150, bbox_inches='tight')
plt.show()
joblib.dump(svd, '../models/svd_model.pkl')

## 3.3 Đánh giá với nhiều giá trị k

In [ ]:
# Test multiple k values
k_values = [3, 5, 7, 8, 10, 12, 15]
results = []

print(f"Testing k = {k_values}...")
print(f"{'k':>4} | {'BIC':>12} | {'AIC':>12} | {'Silhouette':>12}")
print("-"*46)

for k in k_values:
    gmm = GaussianMixture(
        n_components=k,
        covariance_type='diag',
        max_iter=200,
        n_init=3,
        random_state=42
    )
    gmm.fit(X_reduced)
    labels = gmm.predict(X_reduced)
    
    bic = gmm.bic(X_reduced)
    aic = gmm.aic(X_reduced)
    
    # Silhouette trên subset
    if len(X_reduced) > 5000:
        idx = np.random.choice(len(X_reduced), 5000, replace=False)
        sil = silhouette_score(X_reduced[idx], labels[idx])
    else:
        sil = silhouette_score(X_reduced, labels)
    
    results.append({'k': k, 'bic': bic, 'aic': aic, 'silhouette': sil, 'model': gmm})
    print(f"{k:>4} | {bic:>12.1f} | {aic:>12.1f} | {sil:>12.4f}")

In [ ]:
results_df = pd.DataFrame([{k2: v for k2, v in r.items() if k2 != 'model'} for r in results])

# Plot evaluation metrics
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('EM Clustering Evaluation', fontsize=16, fontweight='bold')

axes[0].plot(results_df['k'], results_df['bic'], 'o-', color='#4C72B0', lw=2, ms=8, label='BIC')
axes[0].plot(results_df['k'], results_df['aic'], 's--', color='#DD8452', lw=2, ms=8, label='AIC')
axes[0].set_title('BIC & AIC vs k')
axes[0].set_xlabel('k'); axes[0].set_ylabel('Score'); axes[0].legend()

axes[1].plot(results_df['k'], results_df['silhouette'], 'D-', color='#55A868', lw=2, ms=8)
axes[1].set_title('Silhouette Score vs k')
axes[1].set_xlabel('k'); axes[1].set_ylabel('Silhouette')

# Best k highlight
best_idx = results_df['bic'].idxmin()
best_k = int(results_df.loc[best_idx, 'k'])
axes[0].axvline(best_k, color='red', linestyle=':', lw=2, label=f'Best k={best_k}')
axes[1].axvline(best_k, color='red', linestyle=':', lw=2, label=f'Best k={best_k}')
axes[0].legend(); axes[1].legend()

print(f"\nBest k = {best_k}")
print(f"  BIC = {results_df.loc[best_idx, 'bic']:.1f}")
print(f"  AIC = {results_df.loc[best_idx, 'aic']:.1f}")
print(f"  Silhouette = {results_df.loc[best_idx, 'silhouette']:.4f}")

# Table
axes[2].axis('off')
table_data = results_df[['k', 'bic', 'aic', 'silhouette']].round(2).values.tolist()
table = axes[2].table(
    cellText=table_data,
    colLabels=['k', 'BIC', 'AIC', 'Silhouette'],
    loc='center', cellLoc='center'
)
table.auto_set_font_size(False); table.set_fontsize(10)
axes[2].set_title('Summary Table')

plt.tight_layout()
plt.savefig('../report/images/nb03_em_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.4 Train Model với k tối ưu

In [ ]:
# Train final GMM
best_gmm = results[results_df['bic'].idxmin()]['model']
print(f"Training final GMM with k={best_k}...")

# Predict
labels = best_gmm.predict(X_reduced)
proba  = best_gmm.predict_proba(X_reduced)

# Save
joblib.dump(best_gmm, '../models/gmm_model.pkl')
np.save('../data/processed/cluster_labels.npy', labels)
np.save('../data/processed/X_reduced.npy', X_reduced)
np.save('../data/processed/cluster_proba.npy', proba)

# Summary
unique, counts = np.unique(labels, return_counts=True)
print(f"\nCluster sizes:")
for c, cnt in zip(unique, counts):
    print(f"  Cluster {c}: {cnt:,} items ({cnt/len(labels):.1%})")

## 3.5 Top Keywords mỗi Cluster

In [ ]:
feature_names = np.array(vectorizer.get_feature_names_out())
mat = tfidf_matrix.toarray()

cluster_keywords = {}
print("TOP KEYWORDS PER CLUSTER")
print("="*70)
for c in unique:
    mask = labels == c
    cluster_mean = mat[mask].mean(axis=0)
    top_idx = cluster_mean.argsort()[::-1][:15]
    keywords = feature_names[top_idx].tolist()
    cluster_keywords[c] = keywords
    print(f"Cluster {c:2d} ({counts[c]:5,} items): {', '.join(keywords[:10])}")

import json
with open('../data/processed/cluster_keywords.json', 'w') as f:
    json.dump({str(k): v for k, v in cluster_keywords.items()}, f, indent=2)
print("\nKeywords saved!")

## 3.6 Biểu đồ PCA 2D

In [ ]:
# PCA 2D
print("Running PCA...")
pca = PCA(n_components=2, random_state=42)
n_vis = min(10000, len(X_reduced))
X_pca = pca.fit_transform(X_reduced[:n_vis])
labels_vis = labels[:n_vis]

fig, ax = plt.subplots(figsize=(13, 10))
scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=labels_vis,
                     cmap='tab20', alpha=0.4, s=8)
plt.colorbar(scatter, ax=ax, label='Cluster')
ax.set_title(f'PCA 2D - EM Clusters (k={best_k})', fontsize=14, fontweight='bold')
ax.set_xlabel(f'PC1 (var={pca.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 (var={pca.explained_variance_ratio_[1]:.1%})')
plt.tight_layout()
plt.savefig('../report/images/nb03_pca.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.7 Biểu đồ t-SNE 2D

In [ ]:
# t-SNE 2D
n_tsne = min(5000, len(X_reduced))
idx = np.random.choice(len(X_reduced), n_tsne, replace=False)
print(f"Running t-SNE on {n_tsne} samples (this may take a few minutes)...")

tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000, verbose=1)
X_tsne = tsne.fit_transform(X_reduced[idx])
labels_tsne = labels[idx]

fig, ax = plt.subplots(figsize=(13, 10))
scatter = ax.scatter(X_tsne[:, 0], X_tsne[:, 1], c=labels_tsne,
                     cmap='tab20', alpha=0.5, s=12)
plt.colorbar(scatter, ax=ax, label='Cluster')
ax.set_title(f't-SNE 2D - EM Clusters (k={best_k})', fontsize=14, fontweight='bold')
ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')
plt.tight_layout()
plt.savefig('../report/images/nb03_tsne.png', dpi=150, bbox_inches='tight')
plt.show()
print("Clustering complete!")